# Entraîner Mande-IA v5

Ce cahier entraîne Mande-IA sur une carte graphique gratuite de Google, puis range le résultat dans ton Google Drive.

**Avant de lancer :**
1. Menu **Exécution → Modifier le type d'exécution**, choisis **T4 GPU**, puis **Enregistrer**.
2. Menu **Exécution → Tout exécuter**.
3. Une fenêtre demande l'accès à Google Drive : accepte avec ton compte Google.
4. Laisse l'onglet ouvert, environ 40 minutes. Ne ferme pas Chrome et ne mets pas l'ordinateur en veille.

À la fin, ouvre Google Drive, dossier **Mande-IA-entrainement**. Le guide `entrainement/README.md` du dépôt explique la suite sur le Lenovo.

Base : Llama 3.2 3B Instruct de Meta, la même que Mande-IA v3. Built with Llama.

## 1. Réglages et Google Drive

In [ ]:
NOM = "llama-mande-ia-v5"
BAYELEMABAGA_LIGNES = 3000
FRANCAIS_GENERAL = 500
REPETER_IDENTITE = 4
REPETER_LOCUTEURS = 2
EPOQUES = 1
EXPORTER_GGUF = True
A_VERIFIER = [
    "An ka kɛlɛ ɲuman kɛ an ka dugu la.",
    "N bɛna na sɔgɔma.",
]
DEPOT = "https://raw.githubusercontent.com/Professor-minta/Mande-IA/main"

import os
import torch

if not torch.cuda.is_available():
    raise RuntimeError("Pas de carte graphique. Menu Exécution → Modifier le type d'exécution → T4 GPU, puis relance tout.")
print("Carte graphique :", torch.cuda.get_device_name(0))

from google.colab import drive
drive.mount("/content/drive")
DOSSIER = "/content/drive/MyDrive/Mande-IA-entrainement"
SORTIE = f"{DOSSIER}/{NOM}"
os.makedirs(SORTIE, exist_ok=True)
print("Les résultats iront dans : Mon Drive/Mande-IA-entrainement/" + NOM)

## 2. Installation (3 à 5 minutes)

In [ ]:
!pip install -q unsloth

## 3. Les données
Identité de Mande-IA, phrases de `bambara-demo.js`, contributions validées de Data-Collections (si `data-collections*.csv` est dans Mon Drive/Mande-IA-entrainement), Bayelemabaga de RobotsMali (CC BY-SA 4.0) et conversations en français d'OpenAssistant (Apache 2.0). Les questions de l'examen sont retirées.

In [ ]:
import unsloth
import glob
import json
import random
import re
import unicodedata
import urllib.request

import pandas as pd
from datasets import Dataset

SYSTEM = """Tu es Mande-IA, l'assistant créé au Mali par le Professeur Issa Minta (Minta Services).
Tu réponds en bambara, en français ou en anglais, dans la langue de la personne.
Tu parles simplement, avec des phrases courtes, pour qu'un élève ou un commerçant comprenne.
En bambara, tu écris avec les lettres ɛ, ɔ, ɲ et ŋ.
Si tu ne connais pas un mot en bambara, tu le dis et tu expliques en français simple.
Tu n'inventes jamais un fait, un prix ou un numéro de téléphone.
Liens officiels : https://mintaservices.com, https://ia.mintaservices.com, https://mintaservices.com/academy/"""


HUB = "https://huggingface.co/datasets/{}/resolve/refs%2Fconvert%2Fparquet/{}/train/0000.parquet"


def parquet(depot, config):
    chemin = f"/content/{depot.replace('/', '-')}.parquet"
    if not os.path.exists(chemin):
        urllib.request.urlretrieve(HUB.format(depot, config), chemin)
    return pd.read_parquet(chemin)


def lire(url):
    with urllib.request.urlopen(url, timeout=60) as reponse:
        return reponse.read().decode("utf-8")


def jsonl(texte):
    return [json.loads(ligne) for ligne in texte.splitlines() if ligne.strip()]


def norm(texte):
    texte = unicodedata.normalize("NFC", str(texte)).lower().replace("’", "'")
    return re.sub(r"[\W_]", "", texte)


identite = jsonl(lire(f"{DEPOT}/entrainement/donnees/identite.jsonl"))
examen = jsonl(lire(f"{DEPOT}/entrainement/donnees/examen.jsonl"))
demo = [
    {"bm": bm.replace("\\'", "'"), "fr": fr.replace("\\'", "'")}
    for bm, fr in re.findall(r"bm:\s*'((?:[^'\\]|\\.)*)',\s*fr:\s*'((?:[^'\\]|\\.)*)'", lire(f"{DEPOT}/public/bambara-demo.js"))
]

interdits = {norm(e["question"]) for e in examen}
interdits |= {norm(b) for e in examen for b in e.get("bloquer", [])}
interdits.discard("")
a_verifier = {norm(p) for p in A_VERIFIER}


def autorise(*textes):
    for texte in textes:
        n = norm(texte)
        if n in a_verifier or any(i in n for i in interdits):
            return False
    return True


def conv(question, reponse):
    return {"messages": [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": question},
        {"role": "assistant", "content": reponse},
    ]}


hasard = random.Random(3407)
VERS_BM = ["Traduis en bambara : {}", "Comment dit-on en bambara : « {} »", "Dis-le en bamanankan : {}", "Écris en bambara : {}"]
VERS_FR = ["Traduis en français : {}", "Que veut dire cette phrase en bambara ? « {} »", "Traduis du bambara vers le français : {}", "En français, ça donne quoi : {}"]


def vers_bambara(bm, fr):
    return conv(hasard.choice(VERS_BM).format(fr), bm)


def vers_francais(bm, fr):
    return conv(hasard.choice(VERS_FR).format(bm), fr)


exemples = {}
exemples["identite"] = [conv(x["question"], x["reponse"]) for x in identite if autorise(x["question"], x["reponse"])] * REPETER_IDENTITE

exemples["bambara-demo"] = [
    c for d in demo if autorise(d["bm"], d["fr"])
    for c in (vers_bambara(d["bm"], d["fr"]), vers_francais(d["bm"], d["fr"]))
] * REPETER_LOCUTEURS

paires, vues = [], set()
for chemin in sorted(glob.glob(f"{DOSSIER}/data-collections*.csv")):
    tableau = pd.read_csv(chemin, dtype=str, encoding="utf-8-sig").fillna("")
    for _, ligne in tableau.iterrows():
        bm, fr = ligne.get("bambara", "").strip(), ligne.get("francais", "").strip()
        if ligne.get("statut", "validee") != "validee" or not bm or not fr or not autorise(bm, fr) or norm(bm) in vues:
            continue
        vues.add(norm(bm))
        paires.append((bm, fr))
print(f"Data-Collections : {len(paires)} phrases validées trouvées dans Mon Drive/Mande-IA-entrainement")
exemples["data-collections"] = [c for bm, fr in paires for c in (vers_bambara(bm, fr), vers_francais(bm, fr))] * REPETER_LOCUTEURS


def bambara_moderne(texte):
    t = texte.lower()
    return re.search(r"[ɛɔɲŋ]", t) is not None and re.search(r"\bny|\bbe\b|ʃ", t) is None


bayelemabaga, vues = [], set()
for ligne in parquet("RobotsMaliAI/bayelemabaga", "bam-fr")["translation"]:
    bm, fr = ligne["bam"].strip(), ligne["fr"].strip()
    if 15 <= len(bm) <= 200 and 15 <= len(fr) <= 220 and bambara_moderne(bm) and autorise(bm, fr) and norm(bm) not in vues:
        vues.add(norm(bm))
        bayelemabaga.append((bm, fr))
hasard.shuffle(bayelemabaga)
exemples["bayelemabaga"] = [
    vers_bambara(bm, fr) if i % 2 == 0 else vers_francais(bm, fr)
    for i, (bm, fr) in enumerate(bayelemabaga[:BAYELEMABAGA_LIGNES])
]

general = []
try:
    oa = parquet("OpenAssistant/oasst2", "default")
    oa = oa[(oa["lang"] == "fr") & (~oa["deleted"].astype(bool))]
    questions = oa[(oa["role"] == "prompter") & oa["parent_id"].isna()]
    reponses = oa[(oa["role"] == "assistant") & (oa["rank"] == 0)]
    duos = questions.merge(reponses, left_on="message_id", right_on="parent_id", suffixes=("_q", "_r"))
    for q, r in zip(duos["text_q"], duos["text_r"]):
        if 40 <= len(r) <= 900 and not re.search(r"open ?assistant|chatgpt|openai|laion", q + r, re.I) and autorise(q, r):
            general.append(conv(q.strip(), r.strip()))
    hasard.shuffle(general)
except Exception as erreur:
    print("Conversations en français non chargées :", erreur)
exemples["francais-general"] = general[:FRANCAIS_GENERAL]

donnees = [c for liste in exemples.values() for c in liste]
hasard.shuffle(donnees)
assert all(autorise(*(m["content"] for m in c["messages"][1:])) for c in donnees)

for nom, liste in exemples.items():
    print(f"{nom:18}{len(liste):>7}")
print(f"{'total':18}{len(donnees):>7}")
with open(f"{SORTIE}/donnees.jsonl", "w", encoding="utf-8") as fichier:
    for c in donnees:
        fichier.write(json.dumps(c, ensure_ascii=False) + "\n")

## 4. Le modèle de départ

In [ ]:
from unsloth import FastLanguageModel

LONGUEUR = 1024
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Llama-3.2-3B-Instruct",
    max_seq_length=LONGUEUR,
    load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

## 5. L'entraînement (15 à 25 minutes)

In [ ]:
from trl import SFTConfig, SFTTrainer
from unsloth import is_bfloat16_supported
from unsloth.chat_templates import get_chat_template, train_on_responses_only

tokenizer = get_chat_template(tokenizer, chat_template="llama-3.1")
textes = Dataset.from_list([
    {"text": tokenizer.apply_chat_template(c["messages"], tokenize=False).removeprefix(tokenizer.bos_token)}
    for c in donnees
])

reglages = dict(
    dataset_text_field="text",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    warmup_steps=10,
    num_train_epochs=EPOQUES,
    learning_rate=2e-4,
    logging_steps=25,
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="linear",
    seed=3407,
    output_dir="/content/sorties",
    save_strategy="no",
    report_to="none",
    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),
)
try:
    config = SFTConfig(max_seq_length=LONGUEUR, **reglages)
except TypeError:
    config = SFTConfig(max_length=LONGUEUR, **reglages)
try:
    trainer = SFTTrainer(model=model, processing_class=tokenizer, train_dataset=textes, args=config)
except TypeError:
    trainer = SFTTrainer(model=model, tokenizer=tokenizer, train_dataset=textes, args=config)
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|start_header_id|>user<|end_header_id|>\n\n",
    response_part="<|start_header_id|>assistant<|end_header_id|>\n\n",
)
resultat = trainer.train()
print(f"Entraînement terminé en {resultat.metrics['train_runtime'] / 60:.0f} minutes.")

## 6. L'examen
Mande-IA v5 répond aux questions qu'il n'a jamais vues. Les réponses sont aussi dans `examen.txt`.

In [ ]:
FastLanguageModel.for_inference(model)
copie = []
for e in examen:
    messages = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": e["question"]}]
    entree = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt", return_dict=True).to("cuda")
    sortie = model.generate(**entree, max_new_tokens=200, do_sample=False)
    reponse = tokenizer.decode(sortie[0][entree["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    copie.append(f"Question : {e['question']}\nAttendu : {e['attendu']}\n{NOM} : {reponse}\nNote (2, 1 ou 0) :\n")
    print(copie[-1])
with open(f"{SORTIE}/examen.txt", "w", encoding="utf-8") as fichier:
    fichier.write("\n".join(copie))

## 7. L'adaptateur, petit fichier pour Ollama

In [ ]:
import shutil

model.save_pretrained("/content/adaptateur")
shutil.make_archive(f"{SORTIE}/adaptateur", "zip", root_dir="/content/adaptateur")
with open(f"{SORTIE}/Modelfile", "w", encoding="utf-8") as fichier:
    fichier.write(f"""FROM professor-minta/mande-ia.v3
ADAPTER ./adaptateur
PARAMETER temperature 0.6
PARAMETER num_ctx 4096
SYSTEM \"\"\"{SYSTEM}\"\"\"
""")
print("Adaptateur prêt :", round(os.path.getsize(f"{SORTIE}/adaptateur.zip") / 1e6), "Mo")

## 8. Le modèle complet, en secours (environ 10 minutes, 2 Go)

In [ ]:
GABARIT = """{{ if .System }}<|start_header_id|>system<|end_header_id|>

{{ .System }}<|eot_id|>{{ end }}{{ range .Messages }}{{ if ne .Role "system" }}<|start_header_id|>{{ .Role }}<|end_header_id|>

{{ .Content }}<|eot_id|>{{ end }}{{ end }}<|start_header_id|>assistant<|end_header_id|>

"""

if EXPORTER_GGUF:
    model.save_pretrained_gguf("/content/gguf", tokenizer, quantization_method="q4_k_m")
    fichiers = glob.glob("/content/gguf*/**/*.gguf", recursive=True) + glob.glob("/content/*.gguf")
    choix = [f for f in fichiers if "q4_k_m" in f.lower()] or fichiers
    shutil.copy(max(choix, key=os.path.getmtime), f"{SORTIE}/{NOM}-Q4_K_M.gguf")
    with open(f"{SORTIE}/Modelfile-complet", "w", encoding="utf-8") as fichier:
        fichier.write(f"""FROM ./{NOM}-Q4_K_M.gguf
TEMPLATE \"\"\"{GABARIT}\"\"\"
PARAMETER stop "<|start_header_id|>"
PARAMETER stop "<|end_header_id|>"
PARAMETER stop "<|eot_id|>"
PARAMETER temperature 0.6
PARAMETER num_ctx 4096
SYSTEM \"\"\"{SYSTEM}\"\"\"
""")
    print("Modèle complet prêt :", round(os.path.getsize(f"{SORTIE}/{NOM}-Q4_K_M.gguf") / 1e9, 1), "Go")

In [ ]:
drive.flush_and_unmount()
print("Terminé. Ouvre Google Drive → Mande-IA-entrainement → " + NOM)

## La suite, sur le Lenovo

1. Dans Google Drive, ouvre **Mande-IA-entrainement → llama-mande-ia-v5**.
2. Télécharge **adaptateur.zip**, **Modelfile** et **examen.txt** dans `C:\MandeIA\v5`.
3. Clic droit sur **adaptateur.zip → Extraire tout**, puis **Extraire** : le dossier `adaptateur` apparaît.
4. Dans PowerShell :

```powershell
cd C:\MandeIA\v5
ollama create professor-minta/llama-mande-ia.v5 -f Modelfile
ollama run professor-minta/llama-mande-ia.v5
```

5. Pose les questions de `examen.txt` à v3 puis à v5, et note chaque réponse.

Si v5 répond de façon étrange avec l'adaptateur, télécharge **llama-mande-ia-v5-Q4_K_M.gguf** (2 Go) et **Modelfile-complet**, puis :

```powershell
ollama create professor-minta/llama-mande-ia.v5 -f Modelfile-complet
```